# Clean the refusal notes data (all files)

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from striprtf.striprtf import rtf_to_text
import re
from tqdm import tqdm
#tqdm.pandas() 
import swifter
from collections import defaultdict
import ast
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import gzip
import datetime
import pytz

## Constants

In [ ]:
# For rtf-formatted notes, require these terms to be in header if the only other evidence for a refusal is that 'refus' appears in the section
include_terms = ['IMM', 'VAX', 'VACCINE']
# For rtf-formatted notes, require these terms to not be in header if the only other evidence for a refusal is that 'refus' appears in the section
exclude_terms = ['HISTORY', 'IMMUNOLOGIC', 'GYNECO']

# vaccine-related keywords
vax_names = [
    'rotateq', 'gardasil', 'varivax', 'dtap', 'tdap', 'mmr', 'hep a', 'hep b',
    'influenza', ' flu', 'personal reason', 'religious reason', 'medical reason',
    'caregiver refus', 'shing', 'hepatitis a', 'hepatitis b', 'pneumococcal',
    'vacc', 'vax', 'immu'
]
# vaccine keyword pattern
vax_pattern = re.compile('|'.join(re.escape(word) for word in vax_names), re.IGNORECASE)

# date formatting
date_pattern = r'(\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}-\d{2}-\d{2}\b|\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2}, \d{4})'

# phrases that, if they are the only place where 'refused' appears in the note, bar a note from further consideration
disallowed_phrases = ['refusedweight', 'refusedheight']


## Functions

In [ ]:
# find entries with codes of interest, return rows with and without codes
def process_codes(data):
    if len(data) == 0:
        return data[['patientuid', 'encounterdate']]
    
    # codes of interest
    refusal_source_values = ['V64.00', 'V64.05', 'V64.09', 'V64.06','V64.07','Z28.1', 
                             'Z28.20', 'Z28.21', 'Z28.22', 'Z28.23', 'Z28.24', 'Z28.25',
                             'Z28.26', 'Z28.27', 'Z28.28', 'Z28.29', 
                             'Z28.82', 'Z28.83', 'Z28.89', 'Z28.9'] # from measles_vaccine_dates_data.ipynb

    # combine in a regex
    pattern = '|'.join([f'(?i){val}' for val in refusal_source_values]) 

    # find rows where 'note' contains any of the patterns
    matches = data['note'].str.contains(pattern, na=False)

    # subset into dataframes
    refus_note_codes = data[matches].copy()
    refus_note_no_codes = data[~matches].copy()
    
    refus_note_codes = refus_note_codes[['patientuid', 'encounterdate']]
    
    return (refus_note_codes, refus_note_no_codes)

In [ ]:
# find notes where 'refus' appears in the section headers and the section headers match the include_terms and exclude_terms constants
def find_refus_section_headers(note):
    # find section headers
    headers = [(m.start(), m.group(0).strip()) 
               for m in re.finditer(r'\b([A-Za-z0-9][A-Za-z0-9\s]*?):', note)]

    # find 'refus'
    refus_positions = [m.start() for m in re.finditer(r'refus', note, re.IGNORECASE)]

    result = []

    for pos in refus_positions:
        # get closest header to refus
        header = None
        for h_pos, h_text in headers:
            if h_pos < pos:
                header = h_text
            else:
                break

        if not header:
            continue
            
        if header:
            # check include/exclude terms
            header_upper = header.upper()
            if any(term in header_upper for term in include_terms) and not any(term in header_upper for term in exclude_terms):
                result.append(header)

    return list(set(result))  # Remove duplicates

In [ ]:
# determine if a section contains both 'refus' and a vaccine term
def has_refus_and_vax_same_section(note):
    # Split by colon and check each section
    sections = note.split(':')
    for section in sections:
        if re.search(r'refus', section, re.IGNORECASE) and re.search(vax_pattern, section):
            return True
    return False

In [ ]:
# find dates shortly after 'refus'
def find_filtered_refus_dates(note, lookahead=20):
    # find section headers
    headers = [(m.start(), m.group(0).strip()) 
               for m in re.finditer(r'\b([A-Za-z0-9][A-Za-z0-9\s]*?):', note)]

    # find 'refus'
    refus_positions = [m.start() for m in re.finditer(r'refus', note, re.IGNORECASE)]

    results = []
    date_results = []

    for pos in refus_positions:
        # get closest header to refus
        header = None
        for h_pos, h_text in headers:
            if h_pos < pos:
                header = h_text
            else:
                break

        if not header:
            continue

        # does header match criteria
        header_upper = header.upper()
        include_match = any(term in header_upper for term in include_terms)
        exclude_match = any(term in header_upper for term in exclude_terms)

        if include_match and not exclude_match:
            # append to results
            snippet = note[pos:pos + lookahead]
            date_match = re.search(date_pattern, snippet)
            if date_match:
                date_results.append((header, date_match.group(0)))
                results.append(header)
            else:
                results.append(header)

    return (date_results, results)

In [ ]:
# parse and standardize date strings
def parse_date_string(date_str):
    for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%B %d, %Y", "%b %d, %Y", "%m/%d/%y"):
        try:
            if fmt == "%m/%d/%y":
                date_obj = datetime.datetime.strptime(date_str, fmt)
                if date_obj.year < 100:  # handle 2-digit year (e.g., '18' -> '2018')
                    date_obj = date_obj.replace(year=2000 + date_obj.year)
                return date_obj.replace(tzinfo=pytz.UTC).strftime('%Y-%m-%d %H:%M:%S%z')
            else:
                return datetime.datetime.strptime(date_str, fmt).replace(tzinfo=pytz.UTC).strftime('%Y-%m-%d %H:%M:%S%z')
        except ValueError:
            continue
    return None

In [ ]:
def process_rtf(data):
    if len(data) == 0:
        return data[['patientuid', 'encounterdate']]
    
    # find section headers containing 'refus' matching our include/exclude terms
    #data['refus_sections'] = data['note'].apply(find_refus_section_headers)
    header_subset = data.copy()
    
    res_header = header_subset['note'].apply(find_filtered_refus_dates)
    res_df = pd.DataFrame(res_header.tolist(), columns=['filtered_refus_dates', 'refus_sections'])
    header_subset['filtered_refus_dates'] = res_df['filtered_refus_dates']
    header_subset['refus_sections'] = res_df['refus_sections']
    header_subset = header_subset[header_subset['refus_sections'].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy()
    #header_subset[[(len(x) > 0) for x in header_subset['refus_sections']]] # if the only evidence for a refusal is that refus comes in a certain section, require a vaccine section
    
    
    # determine if a section contains both 'refus' and a vaccine term
    subset_with_refus_and_vax = data[data['note'].apply(has_refus_and_vax_same_section)].copy()
    subset_with_refus_and_vax['filtered_refus_dates'] = subset_with_refus_and_vax['note'].apply(lambda x: [])
    
    
    # join refusals documented in vaccine sections to sections that have refusal and vaccine terminology
    subset_rtf = pd.concat([subset_with_refus_and_vax, header_subset])
    
    # for rows with additional dates, separate out dates
    expanded_rows = []

    for idx, row in subset_rtf.iterrows():
        matches = row['filtered_refus_dates']

        if matches:  # if we have dates
            for section, date_str in matches:
                parsed = parse_date_string(date_str)
                if parsed:
                    new_row = row.copy()
                    new_row['refus_sections'] = section
                    new_row['encounterdate'] = parsed
                    expanded_rows.append(new_row)
        else:
            # No dates — keep row as-is
            new_row = row.copy()
            expanded_rows.append(new_row)

    # expanded dataframe
    expanded_df = pd.DataFrame(expanded_rows)
    
    if len(expanded_df) == 0:
        return subset_rtf[['patientuid', 'encounterdate']]

    else:
        expanded_df = expanded_df[['patientuid', 'encounterdate']]
        expanded_df = expanded_df.drop_duplicates()
        return expanded_df
    
    

In [ ]:
def keep_row_based_on_refu(note):
    if pd.isna(note):
        return True  # keep blank notes

    # all words with refu
    matches = re.findall(r'\b\w*refu\w*\b', note, re.IGNORECASE)

    if not matches:
        return True  # keep if no 'refu' at all

    # check if all matches are in the disallowed list
    normalized = [' '.join(re.findall(r'[a-zA-Z]+', m)).lower() for m in matches]
    return not all(term in disallowed_phrases for term in normalized)


In [ ]:
def extract_relevant_pipe_sections(note):
    if pd.isna(note):
        return []
    
    sections = note.split('|')
    relevant_sections = [
        s.strip() for s in sections
        if re.search(r'refus', s, re.IGNORECASE) and re.search(vax_pattern, s)
    ]
    return relevant_sections


In [ ]:
# process non-rtf notes with sections separated by '|'
def process_sections(data):
    if len(data) == 0:
        return data[['patientuid', 'encounterdate']]
    
    # Filter to notes with more than one '|'
    data['relevant_pipe_sections'] = data['note'].apply(extract_relevant_pipe_sections)

    mask = data['relevant_pipe_sections'].apply(
        lambda sections: not any(s.startswith("PROVIDED VACCINATION") for s in sections))

    # filter to "no_rtf_sections_provid," which are vaccine refusals, and "no_rtf_sections_not_provid," still to examine
    data_not_provid = data[mask].copy()
    data_provid = data[~(mask)].copy()
    
    data_not_provid = data_not_provid[[len(x) > 0 for x in data_not_provid['relevant_pipe_sections']]]
    
    no_rtf_sections = pd.concat([data_not_provid, data_provid])
    no_rtf_sections = no_rtf_sections[['patientuid', 'encounterdate']]
    
    return no_rtf_sections

In [ ]:
# process non-rtf notes that do not have sections separated by '|'
def process_no_sections(data):
    if len(data) == 0:
        return data[['patientuid', 'encounterdate']]
    
    def subset_vax_name_notes(note):
        if re.search(vax_pattern, str(note)):
            return True
        return False

    # filter to notes with vaccine terms
    data_subset = data[data['note'].apply(subset_vax_name_notes)]
    data_subset = data_subset[['patientuid', 'encounterdate']]
    data_subset = data_subset.drop_duplicates()
    
    return data_subset


In [ ]:
def process_non_rtf(data):
    if len(data) == 0:
        return data
    
    # take out rows where 'refu' only appears in the context of refusing a height/weight measurement
    data_wtht = data[data['note'].apply(keep_row_based_on_refu)].copy()
    
    # subset pipe and non-pipe
    data_sections = data_wtht[data_wtht['note'].str.count(r'\|') > 1].copy()
    data_no_sections = data_wtht[~(data_wtht['note'].str.count(r'\|') > 1)].copy()
    
    data_sections_processed = process_sections(data_sections)
    data_no_sections_processed = process_no_sections(data_no_sections)
    
    # put together
    non_rtf_processed = pd.concat([data_sections_processed, data_no_sections_processed])
    return non_rtf_processed

In [ ]:
def process_file_all_methods(data):

    data = data[data['note'] != 'Refused']
    data = data[data['note'] != 'Refused/Deferred/Waived?']
    
    # separate notes with codes and notes without codes
    to_add_codes, no_codes = process_codes(data)
    
    print("did codes")
    # for the notes without codes, separate into rtf and non-rtf formats
    rtf = no_codes[[str(x).lower().startswith('{\\\\rtf') for x in no_codes['note']]]
    no_rtf = no_codes[[not str(x).lower().startswith('{\\\\rtf') for x in no_codes['note']]]
    
    # process rtf formatted notes
    to_add_rtf = process_rtf(rtf)
    print("did rtf")
    
    # separate out pipe and no pipe non-rtf notes
    to_add_no_rtf = process_non_rtf(no_rtf)
    print("did non rtf")
    
    processed_refusals = pd.concat([to_add_codes, to_add_rtf, to_add_no_rtf])
    processed_refusals = processed_refusals.drop_duplicates()
    return processed_refusals

## Read in and process files

In [ ]:
path = '/share/pi/deho/AFC/BQ/Notes_Refusals_0125/'
filenames = os.listdir(path)

In [ ]:
len(filenames)

In [ ]:
# get files that haven't already been made
save_direc = '/share/pi/deho/AFC/mortonc/intermediate/refusals_notes/'
file_head = 'refusal_notes'
files_already_made = os.listdir(save_direc)

processed_filenames = {f.replace(file_head, '') for f in files_already_made if f.startswith(file_head)}
filenames = [f for f in filenames if f not in processed_filenames]

In [ ]:
len(filenames)

In [ ]:
malformed_filenames = ['Notes_Refusals_0125_2019_09.csv.gz','Notes_Refusals_0125_2023_02.csv.gz',
                       'Notes_Refusals_0125_2020_09.csv.gz','Notes_Refusals_0125_2022_04.csv.gz',
                       'Notes_Refusals_0125_2021_01.csv.gz','Notes_Refusals_0125_2021_07.csv.gz',
                      'Notes_Refusals_0125_2021_04.csv.gz', 'Notes_Refusals_0125_2019_12.csv.gz',
                      'Notes_Refusals_0125_2021_11.csv.gz', 'Notes_Refusals_0125_2020_10.csv.gz',
                      'Notes_Refusals_0125_2022_03.csv.gz']
#filenames = [f for f in filenames if f not in malformed_filenames]

In [ ]:
len(filenames)

In [ ]:
np.random.shuffle(filenames)

In [ ]:
filenames

In [ ]:
# process files in parallel
def process_file_parallel(filename):
    full_path = os.path.join(path, filename)
    try:
        df = pd.read_csv(
            full_path,
            engine='python',
            on_bad_lines='skip'  
        )
        #df = pd.read_csv(full_path)
        print(f"Successfully read {full_path}")
        processed_df = df.copy()
        if len(df) != 0:
            processed_df = process_file_all_methods(df)
        print(f"Finished: {filename}")
        processed_df.to_csv(save_direc+file_head+filename, index=False)
        return processed_df, filename  # return processed data and name
    except pd.errors.ParserError as e:
        print(f"Error reading {full_path}: {e}")

In [ ]:
# run in parallel
results = []
with ProcessPoolExecutor() as executor:
    for result in executor.map(process_file_parallel, filenames):
        results.append(result)

In [ ]:
ex = pd.read_csv(save_direc+file_head+'Notes_Refusals_0125_2021_11.csv.gz')

In [ ]:
ex